# Step 4 — Results matrix, plots, claim checklist

Aggregate metrics from Step 3 runs across SNRs, plot SASV-EER vs SNR, and
decide whether the **P2** claim is warranted.

**Claim (only if earned):**  
*SNR-gated enhancement fusion improves spoof-aware speaker verification under additive noise vs raw ECAPA and always-enhance.*

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "noise_gated_lib.py").exists():
    ROOT = ROOT / "replay-cnn-baseline" / "experiments" / "sasv_noise_gated"
sys.path.insert(0, str(ROOT))

from noise_gated_lib import RUNS_DIR, DEFAULT_SNRS_DB, snr_tag, save_json

rows = []
for snr in DEFAULT_SNRS_DB:
    tag = snr_tag(snr)
    for split in ("dev", "eval"):
        path = RUNS_DIR / f"step3_{split}_{tag}" / "metrics.json"
        if not path.exists():
            continue
        payload = json.loads(path.read_text(encoding="utf-8"))
        for system, metrics in payload.items():
            rows.append({
                "split": split,
                "snr": tag,
                "system": system,
                "sasv_eer_percent": metrics.get("sasv_eer_percent"),
                "sv_eer_percent": metrics.get("sv_eer_percent"),
                "spf_eer_percent": metrics.get("spf_eer_percent"),
            })

df = pd.DataFrame(rows)
df

### Table: SASV-EER (%) vs SNR

In [ ]:
if df.empty:
    print("No metrics yet — run notebook 03 for each SNR (and optionally eval).")
else:
    pivot = df[df.split == "dev"].pivot_table(
        index="snr", columns="system", values="sasv_eer_percent"
    )
    display(pivot)
    out_csv = RUNS_DIR / "matrix_sasv_eer_dev.csv"
    pivot.to_csv(out_csv)
    print("wrote", out_csv)

### Plot

In [ ]:
if not df.empty:
    order = [snr_tag(s) for s in DEFAULT_SNRS_DB]
    fig, ax = plt.subplots(figsize=(7, 4))
    sub = df[df.split == "dev"]
    for system in sorted(sub.system.unique()):
        part = sub[sub.system == system].set_index("snr").reindex(order)
        ax.plot(order, part["sasv_eer_percent"], marker="o", label=system)
    ax.set_xlabel("Test condition")
    ax.set_ylabel("SASV-EER (%)")
    ax.set_title("Dev smoke/matrix — replace with locked eval for the paper")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig_path = RUNS_DIR / "sasv_eer_vs_snr_dev.png"
    fig.savefig(fig_path, dpi=150)
    print("wrote", fig_path)
    plt.show()

### Claim checklist (fill after **locked eval**)

Tick only with eval numbers (not smoke):

1. Clean: B1 (ECAPA+AASIST α=0.30) reproduced ≈ **0.83%** SASV-EER  
2. Under noise, **P2 < B0** (gated helps vs raw)  
3. Under noise, **P2 ≤ P3** (gate beats always-enhance) — or explain ties  
4. Ablation table in the paper (gate on/off, CM on/off)  
5. No further tuning on eval after lock

In [ ]:
checklist = {
    "reproduce_B1_clean_eval_0.83": False,
    "P2_beats_B0_under_noise": False,
    "P2_beats_or_matches_P3_under_noise": False,
    "ablations_complete": False,
    "eval_locked_no_retune": False,
    "decision": "PENDING — set True flags only after locked eval runs",
}
save_json(RUNS_DIR / "claim_checklist.json", checklist)
checklist

### Done when
- Matrix CSV + plot exist under `runs/`
- Checklist JSON updated honestly after full eval

If P2 does not win under noise: debug enhancer/gate or **weaken/drop** the claim — do not force the paper.